In [ ]:
# Cài nnU-Net bản chuẩn từ PyPI (Nhanh và sạch)
!pip install -q nnunetv2

# Cài thêm các thư viện bổ trợ nếu thiếu
!pip install -q nibabel matplotlib medpy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 kB 17.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 8.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

# A. Set up

## 1. Giải nén `nnUNet_raw` & `nnUNet_preprocessed`

In [ ]:
import os
import zipfile
from tqdm import tqdm

RAW_ZIP = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_raw/Dataset101_BraTS2020.zip"
PRE_ZIP = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_preprocessed/Dataset101_BraTS2020.zip"

RAW_DIR = "/content"
PRE_DIR = "/content"

def unzip(zip_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        for f in tqdm(z.namelist()):
            z.extract(f, out_dir)

In [ ]:
unzip(RAW_ZIP, RAW_DIR)

100%|██████████| 1848/1848 [03:12<00:00,  9.59it/s]


In [ ]:
# unzip(PRE_ZIP, PRE_DIR)

## 2. Set biến môi trường nnU-Net v2

In [ ]:
import os

os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = "/content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results"

print("nnUNet_raw:", os.environ["nnUNet_raw"])
print("nnUNet_preprocessed:", os.environ["nnUNet_preprocessed"])
print("nnUNet_results:", os.environ["nnUNet_results"])

nnUNet_raw: /content/nnUNet_raw
nnUNet_preprocessed: /content/nnUNet_preprocessed
nnUNet_results: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results


In [ ]:
# Đổi tên file cho đúng chuẩn (giải nén còn thừa .gz)
!for f in /content/nnUNet_raw/Dataset101_BraTS2020/imagesTr/*.nii.gz; do mv "$f" "${f%.gz}"; done

# B. EDL + BASELINE (50 epochs - 5 Folds CV)

## 1. EDL - 250

In [ ]:
# Inject Custom Trainer
import os
import sys
import site
import shutil
import nnunetv2 # Import để tìm đường dẫn cài đặt

# Đường dẫn file gốc của bạn trên Drive
source_trainer = '/content/drive/MyDrive/NCKH/nnUnet/src/trainers/EDLTrainer.py'

# Tìm đường dẫn thư viện nnU-Net v2 trong môi trường Colab
install_path = os.path.dirname(nnunetv2.__file__) # /usr/local/lib/python3.10/dist-packages/nnunetv2
print(install_path)

target_folder = os.path.join(install_path, "training", "nnUNetTrainer")
target_file = os.path.join(target_folder, "EDLTrainer.py")

# Copy file
if os.path.exists(source_trainer):
    shutil.copy(source_trainer, target_file)
    print(f"Đã tiêm EDLTrainer vào: {target_file}")
else:
    print("Lỗi: Không tìm thấy file EDLTrainer trên Drive!")

/usr/local/lib/python3.12/dist-packages/nnunetv2
Đã tiêm EDLTrainer vào: /usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/EDLTrainer.py


In [ ]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/experiment_noise/main.py" --mode edl_250_fixed_split --fold 2 --test_fold 0 --test_only --limit 3

📂 Loading Fixed Test set (Data Fold 0)... | Model: EDL_250_FIXED_SPLIT (Weights Fold 2)
📂 Đã load danh sách Test (fold_0): 74 ca.
🔍 Found 3 cases to process.
Running Noise Experiment:   0% 0/3 [00:00<?, ?it/s]🧪 PROCESSING CASE: BRATS_011

🔥 Running: Gaussian Noise
   ⚡ Simulating Gaussian Noise Level = 0.0...
🔧 Initializing Engine | Mode: EDL...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=True | Bước trượt=0.5
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/EDLTrainer_250epochs_FixedSplit__nnUNetPlans__3d_fullres
🔍 DEBUG: Đang tìm GT cho BRATS_011...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_011.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_011.nii -> KHÔNG

  0% 0/8 [00:00<?, ?it/s]
 12% 1/8 [00:02<00:16,  2.41s/it]
 25% 2/8 [00:02<00:06,  1.14s/it]
 38% 3/8 [00:03<00:04,  1.18it/s]
 50% 4/8 [00:03<00:02,  1.42it/s]
 62% 5/8 [00:04<00:01,  1.59it/s]
 75% 6/8 [00:04

## 2. BASELINE - 250

In [ ]:
import os
import nnunetv2

# 1. Tìm đường dẫn file nnUNetTrainer_Xepochs.py trong môi trường Colab
install_path = os.path.dirname(nnunetv2.__file__)
target_file = os.path.join(install_path, "training", "nnUNetTrainer", "variants", "training_length", "nnUNetTrainer_Xepochs.py")

# 2. Nội dung Class Baseline mới cần tiêm vào
new_trainer_code = """
class nnUNetTrainer_250epochs_FixedSplit(nnUNetTrainer_250epochs):
    def __init__(self, plans: dict, configuration: str, fold: int, dataset_json: dict, device: torch.device = torch.device('cuda')):
        super().__init__(plans, configuration, fold, dataset_json, device)
"""

# 3. Tiến hành tiêm
if os.path.exists(target_file):
    with open(target_file, 'r') as f:
        content = f.read()

    if "class nnUNetTrainer_250epochs_FixedSplit" not in content:
        with open(target_file, 'a') as f:
            f.write("\n" + new_trainer_code + "\n")
        print(f"Đã tiêm thành công nnUNetTrainer_250epochs_FixedSplit vào:\n{target_file}")
    else:
        print("Trainer nnUNetTrainer_250epochs_FixedSplit đã tồn tại sẵn.")
else:
    print(f"Lỗi: Không tìm thấy file gốc tại {target_file}")

# 4. NHẮC NHỞ: Tiêm luôn cả EDLTrainer nếu nhóm chuẩn bị chạy mode EDL
source_trainer = '/content/drive/MyDrive/NCKH/nnUnet/src/trainers/EDLTrainer.py'
target_edl_file = os.path.join(install_path, "training", "nnUNetTrainer", "EDLTrainer.py")

import shutil
if os.path.exists(source_trainer):
    shutil.copy(source_trainer, target_edl_file)
    print(f"Đã tiêm thành công EDLTrainer vào:\n{target_edl_file}")

Trainer nnUNetTrainer_250epochs_FixedSplit đã tồn tại sẵn.
Đã tiêm thành công EDLTrainer vào:
/usr/local/lib/python3.12/dist-packages/nnunetv2/training/nnUNetTrainer/EDLTrainer.py


In [ ]:
!python "/content/drive/MyDrive/NCKH/nnUnet/src/experiment_noise/main.py" --mode baseline_250_fixed_split --fold 2 --test_fold 0 --test_only --limit 3

📂 Loading Fixed Test set (Data Fold 0)... | Model: BASELINE_250_FIXED_SPLIT (Weights Fold 2)
📂 Đã load danh sách Test (fold_0): 74 ca.
🔍 Found 3 cases to process.
Running Noise Experiment:   0% 0/3 [00:00<?, ?it/s]🧪 PROCESSING CASE: BRATS_011

🔥 Running: Gaussian Noise
   ⚡ Simulating Gaussian Noise Level = 0.0...
🔧 Initializing Engine | Mode: BASELINE...
🚀 Initializing nnU-Net Predictor...
   -> Cấu hình: Lật ảnh=True | Bước trượt=0.5
📂 Model loaded from: /content/drive/MyDrive/NCKH/nnUnet/data/nnUNet_results/Dataset101_BraTS2020/nnUNetTrainer_250epochs_FixedSplit__nnUNetPlans__3d_fullres
🔍 DEBUG: Đang tìm GT cho BRATS_011...
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_011.nii.gz -> CÓ
   - Thử: /content/nnUNet_raw/Dataset101_BraTS2020/labelsTr/BRATS_011.nii -> KHÔNG

  0% 0/8 [00:00<?, ?it/s]
 12% 1/8 [00:01<00:11,  1.65s/it]
 25% 2/8 [00:01<00:04,  1.21it/s]
 38% 3/8 [00:02<00:03,  1.47it/s]
 50% 4/8 [00:02<00:02,  1.65it/s]
 62% 5/8 [00:03<00:01,  1.76it/s]
 7